In [283]:
#### Setup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import StaleElementReferenceException
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import SessionNotCreatedException
from selenium.common.exceptions import ElementNotVisibleException
from selenium.common.exceptions import TimeoutException


import csv
import time

import pandas as pd

# read data
data=pd.read_csv("./sampledata-permits.csv")

# list of permits
data_dict=data.to_dict()
permit_list=data["PermitNumber"].to_list() #displays all permits
length=len(data.columns)

# state of inspection
# note that "Not Started "
state = ["Not Started (NS)", "Slab", "Wall", "Roof", "Inspection (INSP)"]

# relevant permits
relevant_permits = ["Footer", "Slab", "Wall Sheathing", "Roof Sheathing", 
                    "Dry In", "Electric Rough", "Framing", "Electric Temporary Service",
                    "Insulation"]

In [284]:
#### Helper Functions
# This function takes in a webElement and gets relevant information from the text
# i.e. the permit type and its status
def GetStatAndPermText(webElement):
    # get the inner html code
    innerHTML = webElement.get_attribute("innerHTML").split('">')
    
    # get the status and permit from the text
    stat = innerHTML[1].split('<')[0]
    perm = innerHTML[2].split('<')[0].split(' (')[0]

    # return the information
    return stat, perm

# This functions takes in a string of text and gets the number of inspection entries
def GetNumRecords(browser):
    
    try:
        # get string containing the number of completed inspections
        completedText = WebDriverWait(browser, 60).until(EC.visibility_of_element_located((By.ID, "ctl00_PlaceHolderMain_InspectionList_lblInspectionCompleted"))).text

        # eliminate the "Complete " and then the number of entries is next
        completedInsp = completedText.split('Completed ')

        # if there is no entry for the number of inspections, 0 are completed
        if (len(completedInsp) == 1):
            return 0

        # get the number of entries from inside the parentheses
        else:
            completedInsp = completedInsp[1].split('\n')
            numRecords = int(completedInsp[0][1:-1])

            # return the number of records
            return numRecords
     
    # if the inspections table does not load, say table is empty
    except (TimeoutException, ElementNotVisibleException):
        return -1
    
    
    
    
# This function tries to move to the next page of inspections
# If it fails, it recursively calls itself to try again until success or the number of times tried reaches the "quit_count"
def RecursivePageTurn(p, browser, delay, permit_number, counter, quit_count=20):
    try:
        # CSS Selector for the given page
        cssSelStr = "#ctl00_PlaceHolderMain_InspectionList_gvListCompleted > tbody > tr.ACA_Table_Pages.ACA_Table_Pages_FontSize > td > table > tbody > tr > td:nth-child(" + str(p+2) + ")"
        
        # web element of the table of inspections
        table = browser.find_element(by=By.CSS_SELECTOR, value=cssSelStr)
        
        # click to the next page
        table.click()
        
        # delay to decrease the chance of exceptions
        time.sleep(delay)

    # if a failure occurs, try again
    # if this doesn't work, add another try and catch block inside this "except"
    # or play with the sleep time
    except (ElementClickInterceptedException, StaleElementReferenceException) as e:
        print("Exception {} {}! On page {} for permit #{}".format(e, counter, p, permit_number))
        
        # continue to try to turn the page
        if (counter < quit_count):
            # recursively call function to continue trying to turn the page
            RecursivePageTurn(p, browser, delay, permit_number, counter+1, quit_count) 
            
        # if too many tries for turning the page were reached, quit and return False for failure
        else:
            print("Page Turn Quit on {} time trying".format(counter))
            return False
          
    # This exception occurs when the format of the page changes due to the list of pages 
    # (i.e. for pages 1-10 there are 12 elements, but for 11 or more, there are only 4 web elements)
    except NoSuchElementException as nsee:
        print("Exception {} {}! No Such Element b/c of page formatting. Continue until found.".format(nsee, counter))
        RecursivePageTurn(p-1, browser, delay, permit_number, counter, quit_count)

    # at this point, success has occurred
    return True

# This function continually/recursively tries to click the search button
def RecursiveSearchButtonClick(browser, counter, quit_count=10):
    # try clicking the search button
    try:
        # find the search button and click it
        permitSearchBtn = browser.find_element(by=By.CLASS_NAME, value="gs_go_img")
        permitSearchBtn.click()
        
    # if it fails, try again    
    except (ElementClickInterceptedException, StaleElementReferenceException):
        print("Exception! Failed to click search button on try {}!".format(counter))
        
        # recursively call function to continue trying to click the search button
        if (counter < quit_count):
            RecursiveSearchButtonClick(browser, counter+1, quit_count)
        
        # if too many tries occur, quit and return False for failure
        else:
            print("Search Button quit on {} time trying".format(counter))
            return False
        
    # at this point, success has occurred    
    return True

### Determine the permits that did not get recorded and write to a specified file
def GetUnsuccessfulPermits(filename, permit_list_use):

    # get the permits that were recorded
    recordedPermits = pd.read_csv("./" + filename + ".csv")
    recordedPermits.drop(labels=recordedPermits.columns.difference(["ID"]), axis=1, inplace=True)

    # get the list of permits used
    triedPermitsList = {"permits": permit_list_use}
    triedPermits = pd.DataFrame(triedPermitsList)
    triedPermits = triedPermits.set_index(keys=triedPermits["permits"])
    triedPermits = triedPermits.rename(columns={"permits":"ID"})

    # get unused permits (due to failure)
    unusedPermits = triedPermits.loc[~triedPermits.index.isin(recordedPermits["ID"])]
    unusedPermits = unusedPermits.reset_index()
    unusedPermits.drop(columns=["permits"], inplace=True, axis=1)
    
    # write to file
    unusedPermits["ID"].to_csv(path_or_buf=filename, index=False)



In [291]:
# This function accesses the search box on the pages and searches for
# a given permit ID
def SearchForPermit(browser, permit_number, quit_count=10):
    # enter permit id into search box
    searchbox = browser.find_element(by=By.ID, value="txtSearchCondition")

    # clear and then enter the permit number into the search bar
    searchbox.clear()
    searchbox.send_keys(str(permit_number))

    # find and click on button to search id
    counterSearch = 1
    searchSuccess = RecursiveSearchButtonClick(browser, counterSearch, quit_count)
    
    # return whether or not the search was successful
    return searchSuccess


# This function goes to the inspections page/tab
def GoToInspections(browser):
    # Go to Inspections for the searched for permit ID
    RecInfoDropdown = browser.find_element(by=By.CSS_SELECTOR, value="[title^='Record Info']")
    RecInfoDropdown.click()

    # keep dropdown open in order to click on inspections option
    browser.implicitly_wait(1)

    # click on "Inspection" option and go to inspections page for given permit id
    InspDropdownBtn = browser.find_element(by=By.CSS_SELECTOR, value="[title^='Inspections']")
    InspDropdownBtn.click()


# This function reads each inspection table and gets each status and permit type    
def GetInspectionInfo(browser, status, permit):
    
    # read status and of permit inspections (this is used to get the number on each page)
    completed = browser.find_element(by=By.ID, value="divInspectionListCompleted")

    # get the table of completed inspections
    inspTable = completed.find_elements(by=By.CLASS_NAME, value="ACA_Width45em")

    # for each completed inspection, get the permit and status
    # done in groups of 5
    for i in range(0,len(inspTable)):

        # get string of text relating to status and permit
        statusText, permitText = GetStatAndPermText(inspTable[i])

        # get the status and permit
        status.append(statusText)
        permit.append(permitText)
        
        
def GetInspectionStatus(status, permit):
    # put status and permit into a pandas dataframe
    data = {"status":status, "permit":permit}
    statusPanda = pd.DataFrame(data)

    # determine status of permit
    passed = statusPanda["status"] == "Pass"

    # begin row to write in csv with permit number
    row = [permit_number]

    # for each relevant permit
    for per in relevant_permits:
        # check if the permit type is in the list
        permitType = statusPanda["permit"] == per

        # if the specified permit type did not pass (resulting dataframe is empty) use "N" for no
        if (statusPanda.loc[((passed) & (permitType))].empty):
            row.append("N")
        # if the specified permit type passed use "Y" for yes
        else:
            row.append("Y")
            
    return row

In [295]:
#### Scrape Data
def ScrapeData(permit_list, state, relevant_permits, overwrite_csv=False, filenameSuccess="PermitStatus", filenameFailure="UnknownPermits", numTryClick=12):
    # get start time
    start_time = time.time()

    # open files in write mode; this overwrites
    if (overwrite_csv):
        # open files to write; overwrites
        file = open("./"+ filenameSuccess +".csv", mode='w')
        fileFailure = open("./" + filenameFailure + ".csv", mode='w')

        # create csv writer for data
        writer = csv.writer(file)
        writerFailure = csv.writer(fileFailure)

        # add column names
        row = ["ID"]
        row = row + relevant_permits + ["State"]
        writer.writerow(row)

    # open files in append mode    
    else:
        # open files to write to using; does not overwrite, just appends
        file = open("./" + filenameSuccess + ".csv", mode='a')
        fileFailure = open("./" + filenameFailure + ".csv", mode='a')

        # create csv writer for data
        writer = csv.writer(file)
        writerFailure = csv.writer(fileFailure)


    # go to start url
    start_url = 'https://secureapps.charlottecountyfl.gov/CitizenAccess/Welcome.aspx?TabName=Home&TabList=Home'    

    # set up web driver
    webDriverPath = "/Users/alexanderwozny/Documents/chromedriver"
    s = Service(webDriverPath)
    browser = webdriver.Chrome(service=s)
    browser.get(start_url)


    # NOTE: IF YOU GET SessionNotCreatedException b/c you have a different
    # chromedriver version than your chrome version,
    # Go to: https://chromedriver.storage.googleapis.com/index.html
    # to download a chromedriver version compatible with your current
    # Chrome version.
    # The error message will say something like: 
    # "Message: session not created: This version of ChromeDriver only supports Chrome version 101
    # Current browser version is 103.0.5060.114"

    success = True

    # for each permit id
    for numIter, permit_number in enumerate(permit_list):

        # search for given permit id on website
        searchSuccess = SearchForPermit(browser=browser, permit_number=permit_number, quit_count=quit_count)

        # if clicking the search button was a success
        if (searchSuccess == True):

            # Go to inspections page/tab
            GoToInspections(browser)

            # get number of records and number of pages
            numRecords = GetNumRecords(browser)

            # if numRecords==-1, the inspection table did not load and couldn't be read
            if (numRecords == -1):
                # move to the next permit
                continue;

            numPages = int(numRecords/5) + 1

            # time delay for preventing stale reference
            delay = 1.5

            # store each completed permit and status
            status = list()
            permit = list()

            # for each page   
            for p in range(1, numPages+1):

                # stop at 10 pages
                if (p > 10):
                    break;

                # Reads each inspection page and get each status and permit type 
                GetInspectionInfo(browser, status, permit)

                # initialize counter for counting the number of tries to turn the page
                counter = 1

                # set success equal to True for if the if statement conditions are not met
                success = True

                # if there is only one page (numRecords > 5) and the current page is not the last, turn the page
                if ((numRecords > 5) & (p != numPages)):
                    success = RecursivePageTurn(p, browser, delay, permit_number, counter, quit_count)


            # if the permit information was successfully retrieved
            if (success == True):
                # get the status of each permit inspection
                row = GetInspectionStatus(status, permit)

                # write a row to the csv
                writer.writerow(row)
            # if the permit information was not successfully retrieved
            else:
                writerFailure.writerow(permit_number)
        else:
            writerFailure.writerow(permit_number)



    # close the csv file
    file.close()
    fileFailure.close()

    # record all the permits that were unsuccessful due to errors and exceptions
    GetUnsuccessfulPermits("UnsuccessfulPermits", permit_list_use)

    # get ending time
    end_time = time.time()

    # print execution time, number of permits checked,
    exe_time = end_time - start_time
    print(f"{exe_time} seconds to execute.")
    print(f"{permit_number} on iteration {numIter}")

In [301]:
ScrapeData(permit_list[0:int(0.01*len(permit_list))+1], state, relevant_permits, overwrite_csv=True)

Exception Message: element click intercepted: Element <td class="aca_pagination_td">...</td> is not clickable at point (592, 618). Other element would receive the click: <iframe title="This is a masked layout for Silverlight control." class="mask_iframe">...</iframe>
  (Session info: chrome=103.0.5060.114)
Stacktrace:
0   chromedriver                        0x0000000107eba079 chromedriver + 4444281
1   chromedriver                        0x0000000107e46403 chromedriver + 3970051
2   chromedriver                        0x0000000107ae1038 chromedriver + 409656
3   chromedriver                        0x0000000107b1e9e5 chromedriver + 661989
4   chromedriver                        0x0000000107b1c593 chromedriver + 652691
5   chromedriver                        0x0000000107b19c24 chromedriver + 642084
6   chromedriver                        0x0000000107b18885 chromedriver + 637061
7   chromedriver                        0x0000000107b0c6a9 chromedriver + 587433
8   chromedriver              

Exception Message: element click intercepted: Element <td class="aca_pagination_td">...</td> is not clickable at point (624, 617). Other element would receive the click: <iframe title="This is a masked layout for Silverlight control." class="mask_iframe">...</iframe>
  (Session info: chrome=103.0.5060.114)
Stacktrace:
0   chromedriver                        0x0000000107eba079 chromedriver + 4444281
1   chromedriver                        0x0000000107e46403 chromedriver + 3970051
2   chromedriver                        0x0000000107ae1038 chromedriver + 409656
3   chromedriver                        0x0000000107b1e9e5 chromedriver + 661989
4   chromedriver                        0x0000000107b1c593 chromedriver + 652691
5   chromedriver                        0x0000000107b19c24 chromedriver + 642084
6   chromedriver                        0x0000000107b18885 chromedriver + 637061
7   chromedriver                        0x0000000107b0c6a9 chromedriver + 587433
8   chromedriver              

Exception Message: element click intercepted: Element <td class="aca_pagination_td aca_pagination_PrevNext">...</td> is not clickable at point (675, 618). Other element would receive the click: <iframe title="This is a masked layout for Silverlight control." class="mask_iframe">...</iframe>
  (Session info: chrome=103.0.5060.114)
Stacktrace:
0   chromedriver                        0x0000000107eba079 chromedriver + 4444281
1   chromedriver                        0x0000000107e46403 chromedriver + 3970051
2   chromedriver                        0x0000000107ae1038 chromedriver + 409656
3   chromedriver                        0x0000000107b1e9e5 chromedriver + 661989
4   chromedriver                        0x0000000107b1c593 chromedriver + 652691
5   chromedriver                        0x0000000107b19c24 chromedriver + 642084
6   chromedriver                        0x0000000107b18885 chromedriver + 637061
7   chromedriver                        0x0000000107b0c6a9 chromedriver + 587433
8   ch

FileNotFoundError: [Errno 2] No such file or directory: './UnsuccessfulPermits.csv'